# 04 — Backtest Model Portfolio

This notebook converts the final trend signal and selected volatility forecasts into monthly portfolio returns. Weights are formed at month-end and applied to the following month's returns.


In [1010]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

try:
    from scipy.optimize import minimize
except ImportError:
    minimize = None

pd.options.display.float_format = "{:,.4f}".format

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
TREND_DIR = RESULTS_DIR / "trend_signal"
VOL_DIR = RESULTS_DIR / "volatility_forecast"
OUT_DIR = RESULTS_DIR / "backtest_model_portfolio"
OUT_DIR.mkdir(parents=True, exist_ok=True)
BALANCED_BENCHMARK_WEIGHTS = {
    "SPY": 0.25,
    "EFA": 0.10,
    "EEM": 0.05,
    "IEF": 0.15,
    "TLT": 0.10,
    "LQD": 0.08,
    "HYG": 0.04,
    "GLD": 0.08,
    "DBC": 0.07,
    "VNQ": 0.05,
    "SHY": 0.03
}

default_layout = dict(
    template="plotly_white",
    height=500,
    margin=dict(t=70, l=60, r=40, b=60),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)


## 1. Load inputs

The backtest uses the monthly asset returns and final trend signal from Notebook 2, the selected monthly volatility forecasts from Notebook 3, and the daily benchmark returns from Notebook 1. Benchmark returns are compounded to monthly frequency before comparison.


In [1011]:
def read_wide_csv(path):
    return (
        pd.read_csv(path, index_col=0, parse_dates=True)
        .sort_index()
        .apply(pd.to_numeric, errors="coerce")
    )


def compound_to_monthly(daily_returns):
    return (1.0 + daily_returns).resample("ME").prod() - 1.0


monthly_returns = read_wide_csv(TREND_DIR / "monthly_returns_aligned.csv")
final_signal = read_wide_csv(TREND_DIR / "final_signal.csv")
vol_forecast = read_wide_csv(VOL_DIR / "final_selected_vol_forecast_monthly_oos.csv")

benchmark_returns_daily = read_wide_csv(DATA_DIR / "benchmark_returns.csv")
benchmark_returns = compound_to_monthly(benchmark_returns_daily)

data_audit = pd.DataFrame({
    "Start": [
        monthly_returns.index.min(),
        benchmark_returns.index.min(),
        final_signal.index.min(),
        vol_forecast.index.min(),
    ],
    "End": [
        monthly_returns.index.max(),
        benchmark_returns.index.max(),
        final_signal.index.max(),
        vol_forecast.index.max(),
    ],
    "Rows": [len(monthly_returns), len(benchmark_returns), len(final_signal), len(vol_forecast)],
    "Columns": [monthly_returns.shape[1], benchmark_returns.shape[1], final_signal.shape[1], vol_forecast.shape[1]],
}, index=[
    "Monthly asset returns",
    "Monthly benchmark returns",
    "Final trend signal",
    "Selected volatility forecasts",
])

display(data_audit)


,Start,End,Rows,Columns
Monthly asset returns,2008-04-30,2026-05-31,218,11
Monthly benchmark returns,2007-04-30,2026-06-30,231,1
Final trend signal,2008-04-30,2026-05-31,218,11
Selected volatility forecasts,2018-01-31,2026-06-30,102,11


## 2. Backtest assumptions

The portfolio is rebalanced monthly. Month-end signals, volatility forecasts, and covariance estimates are treated as information available after that month has closed, so the resulting weights are applied with a one-month lag.

The first evaluated return month is February 2018, because January 2018 is the first month-end with an out-of-sample volatility forecast available for the following month's allocation.


In [1012]:
BACKTEST_START = pd.Timestamp("2018-02-28")
CASH_ASSET = "SHY"
WEIGHT_LAG_MONTHS = 1

RP_WINDOW_MONTHS = 36
RP_MIN_OBS = 24

CORR_WINDOW_MONTHS = 36
CORR_MIN_OBS = 24

VOL_CAP = 0.10
TRANSACTION_COST_BPS = 3.0
RISK_FREE_RATE = 0.0


## 3. Align data and enforce timing

The universe is restricted to assets available in the return, signal, and volatility forecast files. SHY is treated as the defensive cash proxy and excluded from the risky sleeve.


In [1013]:
common_assets = [
    asset for asset in monthly_returns.columns
    if asset in final_signal.columns and asset in vol_forecast.columns
]

if not common_assets:
    raise ValueError("No common assets found across returns, signals, and volatility forecasts.")

cash_asset = CASH_ASSET if CASH_ASSET in common_assets else None
risk_assets = [asset for asset in common_assets if asset != cash_asset]

if not risk_assets:
    raise ValueError("No risk assets available after excluding the cash proxy.")

monthly_returns = monthly_returns.reindex(columns=common_assets).sort_index()
final_signal = final_signal.reindex(columns=common_assets).sort_index()
vol_forecast = vol_forecast.reindex(columns=common_assets).sort_index()

asof_signal = final_signal.reindex(monthly_returns.index).ffill()
asof_vol = vol_forecast.reindex(monthly_returns.index).ffill()

alignment_audit = pd.DataFrame({
    "Included": [len(common_assets), len(risk_assets), 1 if cash_asset is not None else 0]
}, index=["Total assets", "Risk assets", "Cash proxy"])

display(alignment_audit)
print("Risk assets:", risk_assets)
print("Cash proxy:", cash_asset)
print("Backtest start:", BACKTEST_START.date())


,Included
Total assets,11
Risk assets,10
Cash proxy,1


Risk assets: ['SPY', 'EFA', 'EEM', 'IEF', 'TLT', 'LQD', 'HYG', 'GLD', 'DBC', 'VNQ']
Cash proxy: SHY
Backtest start: 2018-02-28


In [1014]:
signal_check = asof_signal.loc[BACKTEST_START:, common_assets].describe().T[["min", "mean", "max"]]
vol_check = asof_vol.loc[BACKTEST_START:, common_assets].describe().T[["min", "mean", "max"]]

print("Signal summary:")
display(signal_check)

print("Volatility forecast summary:")
display(vol_check)


Signal summary:


,min,mean,max
SPY,0.0000,0.7933,1.0000
EFA,0.0000,0.6667,1.0000
EEM,0.0000,0.6000,1.0000
IEF,0.0000,0.5333,1.0000
TLT,0.0000,0.4500,1.0000
LQD,0.0000,0.6367,1.0000
HYG,0.0000,0.7767,1.0000
GLD,0.0000,0.7233,1.0000
DBC,0.0000,0.5933,1.0000
VNQ,0.0000,0.6233,1.0000


Volatility forecast summary:


,min,mean,max
SPY,0.0937,0.1775,0.7701
EFA,0.0964,0.1694,0.7093
EEM,0.1390,0.2078,0.7414
IEF,0.0390,0.0653,0.1487
TLT,0.0958,0.1446,0.4305
LQD,0.0439,0.0723,0.3807
HYG,0.0414,0.0759,0.4143
GLD,0.1067,0.1622,0.4117
DBC,0.0977,0.1768,0.3748
VNQ,0.1101,0.1998,0.9534


## 4. Risk Parity benchmark

The Risk Parity benchmark uses rolling monthly covariance estimates and excludes SHY from the risk-asset universe. The covariance estimate at month-end is applied to the next month's return through the trading lag.


In [1015]:
def inverse_vol_weights(cov):
    vol = pd.Series(np.sqrt(np.diag(cov.values)), index=cov.index).replace(0.0, np.nan)
    inv_vol = (1.0 / vol).replace([np.inf, -np.inf], np.nan).dropna()

    if inv_vol.empty:
        return pd.Series(1.0 / len(cov), index=cov.index)

    return (inv_vol / inv_vol.sum()).reindex(cov.index).fillna(0.0)


def solve_risk_parity_weights(cov):
    cov = cov.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    cov = cov + np.eye(len(cov)) * 1e-10
    fallback = inverse_vol_weights(cov)

    if minimize is None or len(cov) == 1:
        return fallback

    cov_values = cov.values
    n_assets = len(cov)

    def objective(weights):
        portfolio_variance = float(weights @ cov_values @ weights)
        if portfolio_variance <= 0 or not np.isfinite(portfolio_variance):
            return 1e6

        risk_contribution = weights * (cov_values @ weights) / portfolio_variance
        target = 1.0 / n_assets
        return float(((risk_contribution - target) ** 2).sum())

    result = minimize(
        objective,
        x0=fallback.values,
        method="SLSQP",
        bounds=[(0.0, 1.0)] * n_assets,
        constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
        options={"maxiter": 500, "ftol": 1e-12},
    )

    if not result.success or not np.all(np.isfinite(result.x)):
        return fallback

    weights = pd.Series(np.clip(result.x, 0.0, None), index=cov.index)
    return weights / weights.sum()


def build_risk_parity_weights(return_table, risk_assets, window=36, min_obs=24):
    weights = pd.DataFrame(np.nan, index=return_table.index, columns=risk_assets)

    for date in return_table.index:
        history = return_table.loc[:date, risk_assets].tail(window).dropna(how="any")
        if len(history) < min_obs:
            continue

        weights.loc[date, risk_assets] = solve_risk_parity_weights(history.cov())

    return weights


In [1016]:
rp_risk_weights = build_risk_parity_weights(
    monthly_returns,
    risk_assets=risk_assets,
    window=RP_WINDOW_MONTHS,
    min_obs=RP_MIN_OBS,
)

rp_raw_weights = pd.DataFrame(0.0, index=rp_risk_weights.index, columns=common_assets)
rp_raw_weights.loc[:, risk_assets] = rp_risk_weights

rp_raw_weights.tail()


,SPY,EFA,EEM,IEF,TLT,LQD,HYG,GLD,DBC,VNQ,SHY
Date,,,,,,,,,,,
2026-01-31,0.0862,0.0735,0.0628,0.1330,0.0625,0.0979,0.1702,0.0948,0.1646,0.0545,0.0000
2026-02-28,0.0873,0.0686,0.0629,0.1308,0.0577,0.0982,0.1754,0.0974,0.1703,0.0514,0.0000
2026-03-31,0.0825,0.0640,0.0573,0.1338,0.0580,0.0992,0.1731,0.0879,0.1964,0.0478,0.0000
2026-04-30,0.0755,0.0630,0.0533,0.1391,0.0609,0.1038,0.1772,0.0964,0.1844,0.0463,0.0000
2026-05-31,0.0722,0.0638,0.0525,0.1363,0.0588,0.1016,0.1782,0.0948,0.1971,0.0447,0.0000


## 5. Trend + inverse-volatility portfolio

Positive trend scores define the risky sleeve. Within the active sleeve, inverse forecast volatility gives larger weights to lower-risk assets. The average trend score across risky assets determines the total risk budget; the residual is allocated to SHY.


In [1017]:
def clean_signal_scores(signal, assets):
   return signal.reindex(columns=assets).replace([np.inf, -np.inf], np.nan).clip(0.0, 1.0).fillna(0.0)


def build_trend_invvol_weights(
    signal,
    vol,
    all_assets,
    risk_assets,
    cash_asset=None,
    vol_power=0.5,
    rebalance_speed=0.5,
    base_weights=None,
    tactical_weight=0.6,
):
    index = signal.index.intersection(vol.index).sort_values()

    scores = clean_signal_scores(signal.loc[index], risk_assets)

    vol_block = (
        vol.reindex(index=index, columns=risk_assets)
        .replace([np.inf, -np.inf], np.nan)
    )

    inv_vol = 1.0 / vol_block.where(vol_block > 0.0).pow(vol_power)

    raw_scores = scores * inv_vol

    sleeve_weights = (
        raw_scores
        .div(raw_scores.sum(axis=1).replace(0.0, np.nan), axis=0)
        .fillna(0.0)
    )

    risk_budget = np.sqrt(scores.quantile(0.75, axis=1)).clip(0.0, 1.0)

    target_weights = pd.DataFrame(0.0, index=index, columns=all_assets)

    target_weights.loc[:, risk_assets] = sleeve_weights.multiply(risk_budget, axis=0)

    if cash_asset is not None:
        target_weights.loc[:, cash_asset] = (
            1.0 - target_weights[risk_assets].sum(axis=1)
        )

    target_weights = target_weights.clip(lower=0.0)

    if base_weights is not None:
        base = pd.Series(base_weights, index=all_assets).fillna(0.0)
        base = base / base.sum()

        target_weights = (
            tactical_weight * target_weights
            + (1.0 - tactical_weight) * base
        )

    weights = target_weights.ewm(
        alpha=rebalance_speed,
        adjust=False
    ).mean()

    if cash_asset is not None:
        risky_sum = weights[risk_assets].sum(axis=1)
        weights.loc[:, cash_asset] = 1.0 - risky_sum

    return weights.clip(lower=0.0)

In [1018]:
trend_invvol_raw_weights = build_trend_invvol_weights(
    signal=asof_signal,
    vol=asof_vol,
    all_assets=common_assets,
    risk_assets=risk_assets,
    cash_asset=cash_asset,
    base_weights=BALANCED_BENCHMARK_WEIGHTS
)

trend_invvol_raw_weights.loc[BACKTEST_START:].head()


,SPY,EFA,EEM,IEF,TLT,LQD,HYG,GLD,DBC,VNQ,SHY
Date,,,,,,,,,,,
2018-02-28,0.1618,0.1040,0.0733,0.0699,0.0723,0.0789,0.0592,0.0894,0.1043,0.0249,0.1620
2018-03-31,0.1579,0.1047,0.0848,0.0649,0.0765,0.0850,0.0639,0.1168,0.1233,0.0225,0.0998
2018-04-30,0.1647,0.1155,0.0851,0.0625,0.0583,0.0585,0.0715,0.1154,0.1364,0.0212,0.1109
2018-05-31,0.1692,0.1007,0.0736,0.0806,0.0622,0.0453,0.0840,0.1015,0.1216,0.0451,0.1165
2018-06-30,0.1899,0.0876,0.0611,0.0703,0.0706,0.0386,0.1068,0.0865,0.1246,0.0870,0.0770


## 6. Volatility risk cap

The volatility overlay estimates ex-ante portfolio risk using selected asset volatility forecasts and a rolling realized correlation matrix. It only scales down the risky sleeve when estimated volatility exceeds the cap; it does not add leverage when estimated volatility is below the cap.


In [1019]:
def estimate_portfolio_vol(weights, vol, returns, assets, window=36, min_obs=24):
    estimated_vol = pd.Series(np.nan, index=weights.index, name="Estimated_Portfolio_Vol")

    for date in weights.index:
        weight = weights.loc[date, assets].fillna(0.0).astype(float)
        asset_vol = vol.reindex(index=[date], columns=assets).iloc[0].replace([np.inf, -np.inf], np.nan).astype(float)
        asset_vol = asset_vol.where(asset_vol > 0.0)

        if asset_vol.isna().all() or np.isclose(weight.abs().sum(), 0.0):
            continue

        asset_vol = asset_vol.fillna(asset_vol.median())
        history = returns.loc[:date, assets].tail(window).dropna(how="any")

        if len(history) >= min_obs:
            corr_df = history.corr().reindex(index=assets, columns=assets).fillna(0.0)
            corr = corr_df.to_numpy().copy()
            np.fill_diagonal(corr, 1.0)
        else:
            corr = np.eye(len(assets))

        cov_ann = np.outer(asset_vol.values, asset_vol.values) * corr
        portfolio_variance = float(weight.values @ cov_ann @ weight.values)
        estimated_vol.loc[date] = np.sqrt(max(portfolio_variance, 0.0))

    return estimated_vol


def apply_vol_cap(base_weights, estimated_vol, vol_cap, cash_asset=None):
    capped = base_weights.copy().fillna(0.0)
    risk_cols = [col for col in capped.columns if col != cash_asset]

    scale = (vol_cap / estimated_vol.replace(0.0, np.nan)).replace([np.inf, -np.inf], np.nan)
    scale = scale.clip(upper=1.0).fillna(1.0)

    capped.loc[:, risk_cols] = capped[risk_cols].multiply(scale, axis=0)

    if cash_asset is not None:
        capped.loc[:, cash_asset] = 1.0 - capped[risk_cols].sum(axis=1)

    return capped.clip(lower=0.0), scale.rename("Vol_Cap_Scale")


In [1020]:
trend_invvol_est_vol = estimate_portfolio_vol(
    weights=trend_invvol_raw_weights,
    vol=asof_vol,
    returns=monthly_returns,
    assets=common_assets,
    window=CORR_WINDOW_MONTHS,
    min_obs=CORR_MIN_OBS,
)

trend_invvol_cap_raw_weights, vol_cap_scale = apply_vol_cap(
    base_weights=trend_invvol_raw_weights,
    estimated_vol=trend_invvol_est_vol,
    vol_cap=VOL_CAP,
    cash_asset=cash_asset,
)

vol_cap_diagnostics = pd.concat([trend_invvol_est_vol, vol_cap_scale], axis=1)
display(vol_cap_diagnostics.loc[BACKTEST_START:].describe().T)


,count,mean,std,min,25%,50%,75%,max
Estimated_Portfolio_Vol,100.0000,0.0870,0.0239,0.0528,0.0709,0.0873,0.0946,0.2421
Vol_Cap_Scale,100.0000,0.9779,0.0801,0.4131,1.0000,1.0000,1.0000,1.0000


## 7. Compute portfolio returns

Raw month-end weights are lagged by one month before returns are applied. Net returns deduct transaction costs based on one-way turnover. The initial allocation is treated as portfolio funding rather than recurring turnover.


In [1021]:
def lag_weights_to_returns(raw_weights, returns_index, lag_months=1):
    return raw_weights.reindex(returns_index).ffill().shift(lag_months)


def calculate_net_returns(asset_returns, weights, tc_bps=0.0):
    index = weights.dropna(how="all").index.intersection(asset_returns.index)
    weights = weights.loc[index].fillna(0.0)
    asset_returns = asset_returns.reindex(index=index, columns=weights.columns)

    if asset_returns.isna().any().any():
        missing = asset_returns.isna().sum()
        raise ValueError("Missing asset returns detected:\n" + str(missing[missing > 0]))

    tc_rate = tc_bps / 10_000.0
    previous_end_weights = weights.iloc[0].copy()

    net_returns = []
    gross_returns = []
    turnovers = []
    costs = []

    for i, date in enumerate(index):
        weight = weights.loc[date]
        period_return = asset_returns.loc[date]

        turnover = 0.0 if i == 0 else 0.5 * (weight - previous_end_weights).abs().sum()
        gross_return = float((weight * period_return).sum())
        transaction_cost = float(turnover * tc_rate)
        net_return = gross_return - transaction_cost

        denominator = 1.0 + gross_return
        if denominator > 0 and np.isfinite(denominator):
            previous_end_weights = (weight * (1.0 + period_return)) / denominator
            previous_end_weights = previous_end_weights.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        else:
            previous_end_weights = weight.copy()

        gross_returns.append(gross_return)
        net_returns.append(net_return)
        turnovers.append(turnover)
        costs.append(transaction_cost)

    return (
        pd.Series(net_returns, index=index, name="Net_Return"),
        pd.Series(gross_returns, index=index, name="Gross_Return"),
        pd.Series(turnovers, index=index, name="Turnover"),
        pd.Series(costs, index=index, name="Transaction_Cost"),
    )


In [1022]:
raw_weight_sets = {
    "Risk Parity": rp_raw_weights,
    "Trend + Inv Vol": trend_invvol_raw_weights,
    "Trend + Inv Vol + Vol Cap": trend_invvol_cap_raw_weights,
}

trade_weight_sets = {
    name: lag_weights_to_returns(weights, monthly_returns.index, WEIGHT_LAG_MONTHS).loc[BACKTEST_START:]
    for name, weights in raw_weight_sets.items()
}

net_returns = {}
gross_returns = {}
turnover = {}
transaction_costs = {}

for name, weights in trade_weight_sets.items():
    net, gross, to, tc = calculate_net_returns(
        asset_returns=monthly_returns[common_assets],
        weights=weights,
        tc_bps=TRANSACTION_COST_BPS,
    )

    net_returns[name] = net
    gross_returns[name] = gross
    turnover[name] = to
    transaction_costs[name] = tc

strategy_returns = pd.concat(net_returns, axis=1).dropna(how="any")
gross_return_table = pd.concat(gross_returns, axis=1).reindex(strategy_returns.index)
turnover_table = pd.concat(turnover, axis=1).reindex(strategy_returns.index)
transaction_cost_table = pd.concat(transaction_costs, axis=1).reindex(strategy_returns.index)

trade_weight_sets = {
    name: weights.reindex(strategy_returns.index).fillna(0.0)
    for name, weights in trade_weight_sets.items()
}

benchmark_oos = benchmark_returns.reindex(strategy_returns.index).add_prefix("Benchmark: ")
all_return_series = pd.concat([strategy_returns, benchmark_oos], axis=1)

display(strategy_returns.head())
display(strategy_returns.tail())


,Risk Parity,Trend + Inv Vol,Trend + Inv Vol + Vol Cap
Date,,,
2018-02-28,-0.0272,-0.0221,-0.0221
2018-03-31,0.0076,0.0025,0.0025
2018-04-30,-0.0029,-0.0004,-0.0004
2018-05-31,0.0083,0.0051,0.0051
2018-06-30,-0.0038,-0.0076,-0.0076


,Risk Parity,Trend + Inv Vol,Trend + Inv Vol + Vol Cap
Date,,,
2026-01-31,0.0429,0.0311,0.0311
2026-02-28,0.0296,0.0277,0.0277
2026-03-31,-0.0126,-0.0322,-0.0322
2026-04-30,0.0389,0.0418,0.0324
2026-05-31,-0.0001,0.0116,0.0116


## 8. OOS performance diagnostics

Performance is evaluated on the common OOS sample after the trading lag is applied. The main question is whether the adaptive portfolio delivers enough drawdown reduction and risk-adjusted improvement to justify lower upside capture versus Risk Parity.


In [1023]:
def drawdown_series(returns):
    wealth = (1.0 + returns.dropna()).cumprod()
    return wealth / wealth.cummax() - 1.0


def max_drawdown(returns):
    return drawdown_series(returns).min()


def performance_summary(return_table, periods_per_year=12, risk_free_rate=0.0):
    rows = {}

    for name in return_table.columns:
        returns = return_table[name].dropna()
        if returns.empty:
            continue

        ann_return = (1.0 + returns).prod() ** (periods_per_year / len(returns)) - 1.0
        ann_vol = returns.std(ddof=1) * np.sqrt(periods_per_year)
        mdd = max_drawdown(returns)

        rows[name] = {
            "Annual Return": ann_return,
            "Annual Volatility": ann_vol,
            "Sharpe (rf=0)": (ann_return - risk_free_rate) / ann_vol if ann_vol > 0 else np.nan,
            "Max Drawdown": mdd,
            "Calmar": ann_return / abs(mdd) if mdd < 0 else np.nan,
            "Hit Rate": (returns > 0).mean(),
            "Best Month": returns.max(),
            "Worst Month": returns.min(),
        }

    return pd.DataFrame(rows).T


summary_table = performance_summary(all_return_series, risk_free_rate=RISK_FREE_RATE)

summary_table.style.format({
    "Annual Return": "{:.2%}",
    "Annual Volatility": "{:.2%}",
    "Sharpe (rf=0)": "{:.2f}",
    "Max Drawdown": "{:.2%}",
    "Calmar": "{:.2f}",
    "Hit Rate": "{:.2%}",
    "Best Month": "{:.2%}",
    "Worst Month": "{:.2%}",
})


,Annual Return,Annual Volatility,Sharpe (rf=0),Max Drawdown,Calmar,Hit Rate,Best Month,Worst Month
Risk Parity,5.66%,8.13%,0.70,-18.08%,0.31,60.00%,6.26%,-6.54%
Trend + Inv Vol,6.72%,8.01%,0.84,-13.66%,0.49,66.00%,5.18%,-6.24%
Trend + Inv Vol + Vol Cap,6.29%,7.74%,0.81,-12.91%,0.49,66.00%,5.18%,-5.71%
Benchmark: Balanced Benchmark Portfolio,7.97%,9.71%,0.82,-19.07%,0.42,68.00%,6.62%,-7.35%


In [1024]:
turnover_summary = pd.DataFrame({
    "Average Monthly Turnover": turnover_table.mean(),
    "Median Monthly Turnover": turnover_table.median(),
    "Annualized TC Drag": transaction_cost_table.mean() * 12,
    "Max Monthly Turnover": turnover_table.max(),
})

turnover_summary.style.format({
    "Average Monthly Turnover": "{:.2%}",
    "Median Monthly Turnover": "{:.2%}",
    "Annualized TC Drag": "{:.2%}",
    "Max Monthly Turnover": "{:.2%}",
})


,Average Monthly Turnover,Median Monthly Turnover,Annualized TC Drag,Max Monthly Turnover
Risk Parity,1.79%,1.31%,0.01%,8.15%
Trend + Inv Vol,5.88%,4.93%,0.02%,17.09%
Trend + Inv Vol + Vol Cap,7.76%,5.67%,0.03%,51.36%


In [1025]:
wealth = (1.0 + all_return_series.dropna(how="all")).cumprod()

fig = go.Figure()
for name in wealth.columns:
    fig.add_trace(go.Scatter(x=wealth.index, y=wealth[name], mode="lines", name=name))

fig.update_layout(
    **default_layout,
    title="OOS Cumulative Growth of $1",
    xaxis_title="Date",
    yaxis_title="Growth of $1",
)
fig.show()

all_return_series

,Risk Parity,Trend + Inv Vol,Trend + Inv Vol + Vol Cap,Benchmark: Balanced Benchmark Portfolio
Date,,,,
2018-02-28,-0.0272,-0.0221,-0.0221,-0.0304
2018-03-31,0.0076,0.0025,0.0025,0.0020
2018-04-30,-0.0029,-0.0004,-0.0004,-0.0015
2018-05-31,0.0083,0.0051,0.0051,0.0098
2018-06-30,-0.0038,-0.0076,-0.0076,-0.0039
...,...,...,...,...
2026-01-31,0.0429,0.0311,0.0311,0.0306
2026-02-28,0.0296,0.0277,0.0277,0.0270
2026-03-31,-0.0126,-0.0322,-0.0322,-0.0364


In [1026]:
drawdowns = all_return_series.apply(drawdown_series)

fig = go.Figure()
for name in drawdowns.columns:
    fig.add_trace(go.Scatter(x=drawdowns.index, y=drawdowns[name], mode="lines", name=name))

fig.update_layout(
    **default_layout,
    title="OOS Drawdowns",
    xaxis_title="Date",
    yaxis_title="Drawdown",
)
fig.update_yaxes(tickformat=".0%")
fig.show()


## 9. Allocation diagnostics

Allocation diagnostics check whether the realized portfolio behavior matches the design: Risk Parity should remain diversified across risk assets, while the adaptive strategies should move capital into SHY when trend support weakens.


In [1027]:
def allocation_summary(weights_dict, cash_asset=None):
    rows = {}

    for name, weights in weights_dict.items():
        risk_cols = [col for col in weights.columns if col != cash_asset]
        cash_weight = weights[cash_asset] if cash_asset in weights.columns else pd.Series(0.0, index=weights.index)

        rows[name] = {
            "Average Gross Exposure": weights.abs().sum(axis=1).mean(),
            "Max Gross Exposure": weights.abs().sum(axis=1).max(),
            "Average Cash Weight": cash_weight.mean(),
            "Max Cash Weight": cash_weight.max(),
            "Average Active Risk Assets": (weights[risk_cols] > 0.01).sum(axis=1).mean(),
            "Max Single Asset Weight": weights[risk_cols].max(axis=1).max(),
        }

    return pd.DataFrame(rows).T


allocation_table = allocation_summary(trade_weight_sets, cash_asset=cash_asset)
allocation_table.style.format({
    "Average Gross Exposure": "{:.2%}",
    "Max Gross Exposure": "{:.2%}",
    "Average Cash Weight": "{:.2%}",
    "Max Cash Weight": "{:.2%}",
    "Average Active Risk Assets": "{:.1f}",
    "Max Single Asset Weight": "{:.2%}",
})


,Average Gross Exposure,Max Gross Exposure,Average Cash Weight,Max Cash Weight,Average Active Risk Assets,Max Single Asset Weight
Risk Parity,100.00%,100.00%,0.00%,0.00%,10.0,37.82%
Trend + Inv Vol,100.00%,100.00%,8.09%,57.94%,10.0,23.98%
Trend + Inv Vol + Vol Cap,100.00%,100.00%,10.15%,60.27%,10.0,23.18%


In [1028]:
for name, weights in trade_weight_sets.items():
    fig = go.Figure()

    for asset in weights.columns:
        fig.add_trace(
            go.Scatter(
                x=weights.index,
                y=weights[asset],
                mode="lines",
                name=asset,
                stackgroup="one",
            )
        )

    fig.update_layout(
        **default_layout,
        title=f"{name} — Portfolio Weights",
        xaxis_title="Date",
        yaxis_title="Weight",
    )
    fig.update_yaxes(tickformat=".0%")
    fig.show()


In [1029]:
vol_cap_diagnostics_traded = vol_cap_diagnostics.reindex(monthly_returns.index).ffill().shift(WEIGHT_LAG_MONTHS)
vol_cap_diagnostics_traded = vol_cap_diagnostics_traded.reindex(strategy_returns.index)

display(vol_cap_diagnostics_traded.describe().T)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=vol_cap_diagnostics_traded.index,
        y=vol_cap_diagnostics_traded["Vol_Cap_Scale"],
        mode="lines",
        name="Volatility Scale",
    )
)

fig.update_layout(
    **default_layout,
    title="Volatility Cap Scale",
    xaxis_title="Date",
    yaxis_title="Scale",
)
fig.show()


,count,mean,std,min,25%,50%,75%,max
Estimated_Portfolio_Vol,100.0000,0.0866,0.0243,0.0427,0.0706,0.0873,0.0946,0.2421
Vol_Cap_Scale,100.0000,0.9779,0.0801,0.4131,1.0000,1.0000,1.0000,1.0000


## 10. Calendar-year returns

Calendar-year returns provide a final stability check and make it easier to identify which market environments drove the strategy comparison.


In [1030]:
calendar_returns = all_return_series.groupby(all_return_series.index.year).apply(
    lambda x: (1.0 + x).prod() - 1.0
)

calendar_returns.style.format("{:.2%}")


,Risk Parity,Trend + Inv Vol,Trend + Inv Vol + Vol Cap,Benchmark: Balanced Benchmark Portfolio
Date,,,,
2018,-4.86%,-6.62%,-6.62%,-6.20%
2019,16.58%,16.97%,16.97%,19.49%
2020,7.77%,9.57%,7.02%,13.85%
2021,3.88%,10.86%,10.69%,11.10%
2022,-14.39%,-11.30%,-10.53%,-14.22%
2023,8.36%,8.46%,8.40%,12.32%
2024,6.49%,7.48%,7.42%,8.85%
2025,17.12%,15.92%,15.11%,17.66%
2026,10.14%,8.09%,7.10%,8.26%


In [1031]:
strategy_returns.to_csv(OUT_DIR / "strategy_returns_net.csv")
gross_return_table.to_csv(OUT_DIR / "strategy_returns_gross.csv")
turnover_table.to_csv(OUT_DIR / "turnover.csv")
summary_table.to_csv(OUT_DIR / "performance_summary.csv")
allocation_table.to_csv(OUT_DIR / "allocation_summary.csv")

print("Saved backtest outputs to:", OUT_DIR.resolve())


Saved backtest outputs to: /Users/matteomodrian/Documents/Project/results/backtest_model_portfolio
